# Week 2 — Small Language Models: Experiments

**Series:** Weekly AI/ML Research | Week 2/12  
**Blog:** [Small Language Models: Rethinking What Intelligence Actually Requires](https://dev.to/soohan_abbasi)  
**Author:** Soohan Abbasi  

---

## Overview

This notebook contains three experiments run for Week 2 of the AI/ML research series:

- **Experiment 1:** Phi-3 Mini local inference — latency and quality on 3 reasoning tasks
- **Experiment 2:** Llama 3.3 70B (Groq API) vs Phi-3 Mini (local) — head-to-head comparison
- **Experiment 3:** Quantization comparison — FP16 vs 8-bit vs 4-bit NF4

**Hardware:** Kaggle T4 GPU  
**Model:** microsoft/Phi-3-mini-4k-instruct (3.8B)

---

## Setup

In [ ]:
# Install required libraries
!pip install -q transformers==4.44.0 torch accelerate
!pip install -q groq
!pip install -q -U bitsandbytes

---
## Experiment 1: Phi-3 Mini Local Inference

We load Phi-3 Mini (3.8B) locally and test it on three manually crafted prompts covering different reasoning types. The goal is to measure latency, speed (tokens/second), and response quality on a T4 GPU.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import time

MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

# Load tokenizer — converts text to token IDs the model understands
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("Tokenizer loaded! ✅")

# Load model weights (~7GB download on first run, ~5 minutes)
print("\nLoading model... (~5 minutes on first run)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,       # 16-bit precision to reduce memory
    device_map="auto",               # automatically use GPU if available
    trust_remote_code=True,
    attn_implementation="eager",     # disable flash attention for compatibility
)
print(f"\nModel loaded on: {model.device} ✅")

In [ ]:
def generate_response(prompt):
    """Run a single inference pass and return response with latency metrics."""
    messages = [{"role": "user", "content": prompt}]
    
    # Apply the model's chat template to format input correctly
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    # Time the generation
    t0 = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,    # low temperature for deterministic output
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency = time.time() - t0
    
    # Decode only the newly generated tokens, not the input
    gen_ids = output[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(gen_ids, skip_special_tokens=True)
    tok_count = len(gen_ids)
    
    return {
        "response": response,
        "latency_s": round(latency, 2),
        "tok_per_s": round(tok_count / latency, 1)
    }

# Three test prompts covering different reasoning types
# Note: these are manually crafted prompts, not a standard benchmark dataset.
# For rigorous evaluation, GSM8K (math) and HumanEval (code) would be appropriate.
TEST_PROMPTS = [
    {
        "id": "T-001",
        "type": "Math reasoning",
        "prompt": "A store sells apples for $1.20 each and oranges for $0.80 each. "
                  "Maria buys 5 apples and 8 oranges. She pays with a $20 bill. "
                  "How much change does she receive? Show your working."
    },
    {
        "id": "T-002",
        "type": "Code understanding",
        "prompt": "What does this Python function do, and does it contain a bug?\n\n"
                  "def average(nums):\n"
                  "    total = 0\n"
                  "    for n in nums:\n"
                  "        total += n\n"
                  "    return total / len(nums)"
    },
    {
        "id": "T-003",
        "type": "Language reasoning",
        "prompt": "Read the following and identify the core argument in one sentence:\n\n"
                  "'Larger neural networks trained on more data consistently outperform "
                  "smaller ones on general benchmarks. However, recent work shows that "
                  "models trained on curated, high-quality data can exceed the performance "
                  "of much larger models trained on raw internet text, suggesting that "
                  "data quality may matter more than quantity past a certain scale.'"
    },
]

print("=" * 60)
print("  Experiment 1 — Phi-3 Mini Inference")
print("=" * 60)

exp1_results = []
for case in TEST_PROMPTS:
    print(f"\n[{case['id']}] {case['type']}")
    result = generate_response(case["prompt"])
    exp1_results.append({**case, **result})
    print(f"Latency  : {result['latency_s']}s")
    print(f"Speed    : {result['tok_per_s']} tok/s")
    print(f"Response : {result['response']}")
    print("-" * 60)

print("\nSUMMARY")
print(f"{'ID':<8} {'Type':<22} {'Latency':>10} {'Tok/s':>8}")
print("-" * 50)
for r in exp1_results:
    print(f"{r['id']:<8} {r['type']:<22} {r['latency_s']:>8.1f}s {r['tok_per_s']:>8.1f}")

---
## Experiment 2: Llama 3.3 70B (Cloud) vs Phi-3 Mini (Local)

We compare Phi-3 Mini running locally against Llama 3.3 70B accessed via the Groq API. The same three prompts are used for a direct comparison of latency and output quality.

**Note:** Replace `YOUR_GROQ_KEY` with your actual Groq API key from [console.groq.com](https://console.groq.com).

In [ ]:
from groq import Groq
import time

# Initialize Groq client — free API for Llama 3.3 70B
GROQ_KEY = "YOUR_GROQ_KEY"   # replace with your key
groq_client = Groq(api_key=GROQ_KEY)

# Same test cases used in Experiment 1
TEST_CASES = TEST_PROMPTS

def query_llama(prompt):
    """Query Llama 3.3 70B via Groq cloud API and measure latency."""
    t0 = time.time()
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=256,
        temperature=0.1,
    )
    latency = time.time() - t0
    return {
        "response": response.choices[0].message.content,
        "latency_s": round(latency, 2),
    }

def query_phi3(prompt):
    """Run Phi-3 Mini inference locally and return response with metrics."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency = time.time() - t0
    
    # Decode only newly generated tokens
    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(gen_ids, skip_special_tokens=True)
    tok_count = len(gen_ids)
    
    return {
        "response": response,
        "latency_s": round(latency, 2),
        "tok_per_s": round(tok_count / latency, 1),
    }

print("=" * 65)
print("  Experiment 2 — Llama 3.3 70B (Groq) vs Phi-3 Mini (Local)")
print("=" * 65)

exp2_results = []
for case in TEST_CASES:
    print(f"\n[{case['id']}] {case['type']}")
    print(f"Prompt: {case['prompt'][:70]}...")

    # Query large model via cloud API
    llama = query_llama(case["prompt"])
    print(f"\n  Llama 3.3 70B  | {llama['latency_s']}s | Cloud")
    print(f"  {llama['response'][:200]}...")

    # Query small model locally
    phi3 = query_phi3(case["prompt"])
    print(f"\n  Phi-3 Mini     | {phi3['latency_s']}s | Local | {phi3['tok_per_s']} tok/s")
    print(f"  {phi3['response'][:200]}...")

    exp2_results.append({
        "id": case["id"],
        "type": case["type"],
        "llama_latency": llama["latency_s"],
        "phi3_latency": phi3["latency_s"],
        "phi3_speed": phi3["tok_per_s"],
    })

print("\n" + "=" * 65)
print("SUMMARY")
print(f"{'ID':<8} {'Type':<22} {'Llama lat':>10} {'Phi-3 lat':>10} {'Phi3 tok/s':>11}")
print("=" * 65)
for r in exp2_results:
    print(f"{r['id']:<8} {r['type']:<22} {r['llama_latency']:>8.1f}s "
          f"{r['phi3_latency']:>8.1f}s {r['phi3_speed']:>10.1f}")

---
## Experiment 3: Quantization Comparison

We load Phi-3 Mini three times with different precision settings and compare VRAM usage, latency, and output quality. This shows the practical tradeoff of running a model with reduced numerical precision.

| Config | Expected VRAM | Expected quality loss |
|---|---|---|
| FP16 (baseline) | ~4 GB | None |
| 8-bit | ~2 GB | Negligible |
| 4-bit NF4 | ~1 GB | Minor on complex tasks |

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import time

MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

# Reload tokenizer for this experiment
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Fixed prompt for consistent comparison across all three configurations
QUANT_PROMPT = "Explain the difference between supervised and unsupervised learning in two concise paragraphs."

# Three quantization configurations to compare
CONFIGS = [
    {
        "name": "FP16 (baseline)",
        "kwargs": {"torch_dtype": torch.float16}   # standard 16-bit, no quantization
    },
    {
        "name": "8-bit",
        "kwargs": {"quantization_config": BitsAndBytesConfig(load_in_8bit=True)}  # halves VRAM
    },
    {
        "name": "4-bit NF4",
        "kwargs": {"quantization_config": BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",             # NormalFloat4 — best quality at 4-bit
            bnb_4bit_use_double_quant=True         # quantize the quantization constants too
        )}
    },
]

print("=" * 60)
print("  Experiment 3 — Quantization Comparison")
print("=" * 60)

exp3_results = []
for cfg in CONFIGS:
    print(f"\nLoading {cfg['name']}...")
    model_q = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="eager",
        **cfg["kwargs"]
    )

    # Reset VRAM counter before inference to get accurate peak usage
    torch.cuda.reset_peak_memory_stats()

    msgs = [{"role": "user", "content": QUANT_PROMPT}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model_q.device)

    t0 = time.time()
    with torch.no_grad():
        out = model_q.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency = time.time() - t0

    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(gen_ids, skip_special_tokens=True)
    tok_count = len(gen_ids)
    mem_gb = torch.cuda.max_memory_allocated() / 1e9   # peak VRAM in GB

    exp3_results.append({
        "config": cfg["name"],
        "latency_s": round(latency, 2),
        "tok_per_s": round(tok_count / latency, 1),
        "memory_gb": round(mem_gb, 2),
        "response": response
    })

    print(f"  VRAM    : {mem_gb:.2f} GB")
    print(f"  Latency : {latency:.1f}s")
    print(f"  Speed   : {tok_count / latency:.1f} tok/s")
    print(f"  Response: {response[:200]}...")

    # Free GPU memory before loading next configuration
    del model_q
    torch.cuda.empty_cache()

print("\n" + "=" * 60)
print("SUMMARY")
print(f"{'Config':<20} {'VRAM':>8} {'Latency':>10} {'Tok/s':>8}")
print("=" * 60)
for r in exp3_results:
    print(f"{r['config']:<20} {r['memory_gb']:>6.2f}GB {r['latency_s']:>8.1f}s {r['tok_per_s']:>8.1f}")

---
## Results Visualization

Generate a combined chart for all three experiments.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('#0d1117')

colors = ['#534AB7', '#1D9E75', '#BA7517']
tasks = ['Math\nReasoning', 'Code\nUnderstanding', 'Language\nReasoning']

# Chart 1: Experiment 1 — Phi-3 latency per task
ax1 = axes[0, 0]
ax1.set_facecolor('#161b22')
latencies = [r['latency_s'] for r in exp1_results]
bars = ax1.bar(tasks, latencies, color=colors, width=0.5, zorder=3)
ax1.set_title('Exp 1 — Phi-3 Mini Latency per Task', color='white', fontsize=11, pad=10)
ax1.set_ylabel('Latency (seconds)', color='#9c9a92', fontsize=9)
ax1.tick_params(colors='#9c9a92', labelsize=8)
ax1.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
ax1.yaxis.grid(True, color='#21262d', zorder=0)
for bar, val in zip(bars, latencies):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val}s', ha='center', color='white', fontsize=9, fontweight='bold')

# Chart 2: Experiment 2 — Llama vs Phi-3 latency
ax2 = axes[0, 1]
ax2.set_facecolor('#161b22')
x = np.arange(len(tasks))
w = 0.35
llama_lat = [r['llama_latency'] for r in exp2_results]
phi3_lat  = [r['phi3_latency'] for r in exp2_results]
b1 = ax2.bar(x - w/2, llama_lat, w, label='Llama 3.3 70B (Cloud)', color='#534AB7', zorder=3)
b2 = ax2.bar(x + w/2, phi3_lat,  w, label='Phi-3 Mini (Local)',    color='#1D9E75', zorder=3)
ax2.set_title('Exp 2 — Llama 3.3 70B vs Phi-3 Mini Latency', color='white', fontsize=11, pad=10)
ax2.set_ylabel('Latency (seconds)', color='#9c9a92', fontsize=9)
ax2.set_xticks(x)
ax2.set_xticklabels(tasks, fontsize=8)
ax2.tick_params(colors='#9c9a92', labelsize=8)
ax2.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
ax2.yaxis.grid(True, color='#21262d', zorder=0)
ax2.legend(fontsize=8, facecolor='#21262d', labelcolor='white', edgecolor='none')
for bar, val in zip(b1, llama_lat):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{val}s', ha='center', color='white', fontsize=8, fontweight='bold')
for bar, val in zip(b2, phi3_lat):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{val}s', ha='center', color='white', fontsize=8, fontweight='bold')

# Chart 3: Experiment 3 — VRAM usage
ax3 = axes[1, 0]
ax3.set_facecolor('#161b22')
configs  = [r['config'] for r in exp3_results]
vram     = [r['memory_gb'] for r in exp3_results]
bars3 = ax3.bar(configs, vram, color=colors, width=0.5, zorder=3)
ax3.set_title('Exp 3 — Quantization VRAM Usage', color='white', fontsize=11, pad=10)
ax3.set_ylabel('VRAM (GB)', color='#9c9a92', fontsize=9)
ax3.tick_params(colors='#9c9a92', labelsize=8)
ax3.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
ax3.yaxis.grid(True, color='#21262d', zorder=0)
for bar, val in zip(bars3, vram):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
             f'{val} GB', ha='center', color='white', fontsize=9, fontweight='bold')

# Chart 4: Experiment 3 — Speed comparison
ax4 = axes[1, 1]
ax4.set_facecolor('#161b22')
speeds = [r['tok_per_s'] for r in exp3_results]
bars4 = ax4.bar(configs, speeds, color=colors, width=0.5, zorder=3)
ax4.set_title('Exp 3 — Quantization Speed (tok/s)', color='white', fontsize=11, pad=10)
ax4.set_ylabel('Tokens per second', color='#9c9a92', fontsize=9)
ax4.tick_params(colors='#9c9a92', labelsize=8)
ax4.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
ax4.yaxis.grid(True, color='#21262d', zorder=0)
for bar, val in zip(bars4, speeds):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{val}', ha='center', color='white', fontsize=9, fontweight='bold')

plt.suptitle('Week 2 — Phi-3 Mini: All Experiment Results',
             color='white', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('week2_all_results.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117', edgecolor='none')
plt.show()
print("Saved: week2_all_results.png ✅")